# Hotel Booking Cancellation Predictor — EDA & Model Training

**Run this notebook in Google Colab.**

This notebook:
1. Loads the real Kaggle "Hotel Booking Demand" dataset (you upload it)
2. Explores the data (EDA)
3. Cleans + engineers features
4. Trains and compares classification models
5. Evaluates the best model
6. Saves the trained pipeline for use in the Streamlit app

Dataset source: https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand

## 1. Setup

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## 2. Upload the dataset

Download `hotel_bookings.csv` from Kaggle first, then run the cell below and
upload it when prompted. (Alternatively, mount Google Drive if you've stored
the file there.)

In [ ]:
from google.colab import files

uploaded = files.upload()  # select hotel_bookings.csv
DATA_PATH = list(uploaded.keys())[0]

In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

## 3. Exploratory Data Analysis

In [ ]:
df.info()
print("\nMissing values:\n", df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
cancel_rate = df["is_canceled"].mean()
print(f"Overall cancellation rate: {cancel_rate:.1%}")

sns.countplot(data=df, x="is_canceled")
plt.title("Booking Outcome Distribution (0 = Show, 1 = Canceled)")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="is_canceled", y="lead_time")
plt.title("Lead Time vs Cancellation")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=df, x="deposit_type", y="is_canceled")
plt.title("Cancellation Rate by Deposit Type")
plt.ylabel("Cancellation Rate")
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=df, x="market_segment", y="is_canceled")
plt.title("Cancellation Rate by Market Segment")
plt.xticks(rotation=30)
plt.ylabel("Cancellation Rate")
plt.show()

**Key EDA takeaways to note in your presentation:**
- Longer lead time is generally associated with higher cancellation rates.
- Non-refundable deposits show a very different cancellation pattern than
  "No Deposit" bookings — worth calling out explicitly (and worth an SHAP
  check later, since this is a known counter-intuitive signal in this
  dataset).
- Certain market segments and distribution channels cancel far more than
  others.

## 4. Feature Engineering & Cleaning

In [ ]:
MONTH_MAP = {
    "January": 1, "February": 2, "March": 3, "April": 4, "May": 5, "June": 6,
    "July": 7, "August": 8, "September": 9, "October": 10, "November": 11, "December": 12,
}

def clean_and_engineer(df):
    df = df.copy()
    df["children"] = df["children"].fillna(0)
    df["country"] = df["country"].fillna("Unknown")
    df["agent"] = df["agent"].fillna(0)
    df["company"] = df["company"].fillna(0)

    df = df[(df["adults"] + df["children"] + df["babies"]) > 0]

    # Drop leakage columns -- these directly encode the outcome
    df = df.drop(columns=[c for c in ["reservation_status", "reservation_status_date"] if c in df.columns])

    df["arrival_month_num"] = df["arrival_date_month"].map(MONTH_MAP)
    df["arrival_week_number"] = df["arrival_date_week_number"]
    df["stays_total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
    df["room_type_mismatch"] = (df["reserved_room_type"] != df["assigned_room_type"]).astype(int)

    return df

df_clean = clean_and_engineer(df)
df_clean.shape

In [ ]:
MODEL_FEATURES = [
    "hotel", "lead_time", "arrival_month_num", "arrival_week_number",
    "stays_total_nights", "adults", "children", "babies", "meal",
    "market_segment", "distribution_channel", "is_repeated_guest",
    "previous_cancellations", "previous_bookings_not_canceled",
    "booking_changes", "deposit_type", "days_in_waiting_list",
    "customer_type", "adr", "required_car_parking_spaces",
    "total_of_special_requests", "room_type_mismatch",
]
TARGET = "is_canceled"

CATEGORICAL_FEATURES = ["hotel", "meal", "market_segment", "distribution_channel", "deposit_type", "customer_type"]
NUMERIC_FEATURES = [f for f in MODEL_FEATURES if f not in CATEGORICAL_FEATURES]

X = df_clean[MODEL_FEATURES]
y = df_clean[TARGET]
X.head()

## 5. Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

## 6. Build Preprocessing + Model Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, accuracy_score, f1_score

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
])

candidates = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=300, max_depth=12, random_state=42),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42),
}

results = {}
for name, model in candidates.items():
    pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = pipe.predict(X_test)
    auc = roc_auc_score(y_test, proba)
    f1 = f1_score(y_test, pred)
    results[name] = {"pipeline": pipe, "auc": auc, "f1": f1}
    print(f"{name}: AUC={auc:.4f}, F1={f1:.4f}")

## 7. Select Best Model & Evaluate

In [ ]:
best_name = max(results, key=lambda k: results[k]["auc"])
best_pipe = results[best_name]["pipeline"]
print(f"Best model: {best_name}")

y_pred = best_pipe.predict(X_test)
y_proba = best_pipe.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title(f"Confusion Matrix — {best_name}")
plt.show()

In [ ]:
from sklearn.metrics import RocCurveDisplay

RocCurveDisplay.from_predictions(y_test, y_proba)
plt.title(f"ROC Curve — {best_name}")
plt.show()

## 8. Save the Trained Pipeline

In [ ]:
import joblib
import json

joblib.dump(best_pipe, "cancellation_model.pkl")

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "roc_auc": roc_auc_score(y_test, y_proba),
    "n_train": len(X_train),
    "n_test": len(X_test),
    "best_model": best_name,
}
with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

files.download("cancellation_model.pkl")
files.download("metrics.json")
print("Download these two files and place them in your repo's models/ folder.")

## 9. Next Steps

1. Move `cancellation_model.pkl` and `metrics.json` into `models/` in your
   local repo (or commit them directly if working in Colab connected to
   GitHub).
2. Run `streamlit run app.py` locally to test the full app against this
   real-data-trained model.
3. Push to GitHub and deploy on Streamlit Community Cloud.